In [3]:
# Imports

#Checking the installed Java version
!java -version
!pip install pyspark 
# Install Java 17
!sudo apt-get update
!sudo apt-get install -y openjdk-17-jdk-headless

!java -version

# Set JAVA_HOME to Java 17
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

from pyspark.sql import SparkSession

spark = SparkSession.builder\
        .master("local[*]")\
        .appName("ML") \
        .getOrCreate()
print("Spark ready:", spark.version)

spark.sparkContext.setLogLevel("ERROR")

import warnings
warnings.filterwarnings("ignore")

from pyspark.sql.functions import col, count
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import RandomForestClassifier, LogisticRegression
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

openjdk version "17.0.17" 2025-10-21
OpenJDK Runtime Environment (build 17.0.17+10-Ubuntu-124.04)
OpenJDK 64-Bit Server VM (build 17.0.17+10-Ubuntu-124.04, mixed mode, sharing)
Hit:1 https://packages.cloud.google.com/apt cloud-sdk InRelease
Hit:2 https://download.docker.com/linux/ubuntu noble InRelease                 
Get:3 https://cli.github.com/packages stable InRelease [3917 B]                
Hit:4 https://us-east-1.ec2.archive.ubuntu.com/ubuntu noble InRelease          
Hit:5 https://us-east-1.ec2.archive.ubuntu.com/ubuntu noble-updates InRelease  
Hit:6 https://cloud.archive.ubuntu.com/ubuntu noble InRelease                  
Hit:7 https://us-east-1.ec2.archive.ubuntu.com/ubuntu noble-backports InRelease
Hit:8 https://cloud.archive.ubuntu.com/ubuntu noble-updates InRelease          
Hit:9 https://cloud.archive.ubuntu.com/ubuntu noble-backports InRelease        
Hit:10 http://deb.wakemeops.com/wakemeops stable InRelease                     
Get:11 https://cloud.archive.ubuntu.com

In [4]:
# Load Data

df = spark.read.parquet("/teamspace/studios/this_studio/cleaned_flights_parquet")
df.printSchema()

root
 |-- FL_DATE: date (nullable = true)
 |-- OP_CARRIER: string (nullable = true)
 |-- OP_CARRIER_FL_NUM: integer (nullable = true)
 |-- ORIGIN: string (nullable = true)
 |-- DEST: string (nullable = true)
 |-- CRS_DEP_TIME: integer (nullable = true)
 |-- DEP_TIME: float (nullable = true)
 |-- DEP_DELAY: float (nullable = true)
 |-- TAXI_OUT: float (nullable = true)
 |-- WHEELS_OFF: float (nullable = true)
 |-- WHEELS_ON: float (nullable = true)
 |-- TAXI_IN: float (nullable = true)
 |-- CRS_ARR_TIME: integer (nullable = true)
 |-- ARR_TIME: float (nullable = true)
 |-- ARR_DELAY: float (nullable = true)
 |-- CANCELLED: float (nullable = true)
 |-- DIVERTED: float (nullable = true)
 |-- CRS_ELAPSED_TIME: float (nullable = true)
 |-- ACTUAL_ELAPSED_TIME: float (nullable = true)
 |-- AIR_TIME: float (nullable = true)
 |-- DISTANCE: float (nullable = true)
 |-- CARRIER_DELAY: float (nullable = true)
 |-- WEATHER_DELAY: float (nullable = true)
 |-- NAS_DELAY: float (nullable = true)
 |--

In [5]:
# Make a route feature to find congested routes and more accurately predict delays

route_counts = (
    df.groupBy("ORIGIN", "DEST")
      .agg(count("*").alias("ROUTE_FREQ"))
)

df = df.join(route_counts, on=["ORIGIN", "DEST"], how="left")

In [6]:
# Feature Columns
# We only use features that can be known before a flight, otherwise the prediction would mean nothing.

cat_cols = ["OP_CARRIER", "ORIGIN", "DEST"]
num_cols = ["CRS_DEP_TIME", "CRS_ARR_TIME", "DISTANCE", "Month", "ROUTE_FREQ"]

In [7]:
# Index + One-Hot Encode to index strings and turn them into usable vectors
# handleInvalid="keep" ensures that the model doesnt crash if a new airport shows up in the 2018 data (which did happen)

indexers = [
    StringIndexer(inputCol=c, outputCol=c+"_idx", handleInvalid="keep")
    for c in cat_cols
]

encoders = [
    OneHotEncoder(inputCols=[c+"_idx"], outputCols=[c+"_vec"], handleInvalid="keep")
    for c in cat_cols
]

In [8]:
# Assemble Features

# Logistic regression assembler is kept simple as our main model will be Random Forest, this is just for comparison
assembler_lr = VectorAssembler(
    inputCols=["ROUTE_FREQ", "DISTANCE", "CRS_DEP_TIME"],
    outputCol="features"
)

# Random Forest assembler is more complex as it can work with higher dimensionality
assembler_rf = VectorAssembler(
    inputCols=[c+"_vec" for c in cat_cols] + num_cols,
    outputCol="features",
    handleInvalid="keep"
)

In [9]:
# Time Split
# The models are trained on 2017 data and tested on 2018 data instead of a random split because flight data has seasonality and trends, which the model should learn to predict accurately

train = df.filter(col("Year") == 2017)
test  = df.filter(col("Year") == 2018)

In [10]:
# Random Forest Model

rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="IsDelayed",
    numTrees=100,
    maxDepth=8
)

In [11]:
# Model selection and hyperparameter tuning

# Select parameters of the Random Forest models we test
param_grid = (
    ParamGridBuilder()
    .addGrid(rf.numTrees, [50, 100])
    .addGrid(rf.maxDepth, [6, 10])
    .build()
)

# Measure model performance
evaluator = BinaryClassificationEvaluator(
    labelCol="IsDelayed",
    metricName="areaUnderROC"
)


In [12]:
# Logistic regression (not our main model, just to compare to random forest)

train_lr = assembler_lr.transform(train)
test_lr = assembler_lr.transform(test)

lr = LogisticRegression(
    featuresCol="features",
    labelCol="IsDelayed",
    maxIter=20
)

lr_model = lr.fit(train_lr)
lr_pred = lr_model.transform(test_lr)

lr_auc = evaluator.evaluate(lr_pred)

print(f"LR AUC: {lr_auc}")

LR AUC: 0.6033927570745347


In [13]:
# Random Forest Pipeline

pipeline = Pipeline(stages=indexers + encoders + [assembler_rf, rf])

In [14]:
# Cross Validator 
# Repeatedly splits training data into train + validation parts to avoid lucky/unlucky splits
cv = CrossValidator(
    estimator=pipeline,
    estimatorParamMaps=param_grid,
    evaluator=evaluator,
    numFolds=2,       
    parallelism=2
)

In [15]:
# Train
# This takes around 90 minutes to run. 
# We ignored warnings earlier for this training, because otherwise almost every step gets a MemoryStore warning, bloating the output. 
# These warnings come up because the tools we are using are not made for this amount of data, the only thing it actually causes is longer runtime, which is fine. If we had actual company level servers, it would be fine

cv_model = cv.fit(train)

In [16]:
# Predict

rf_pred = cv_model.transform(test)

In [17]:
# Metrics
# We mainly care about the AUC, because accuracy and F1 will always be relatively high since most flights are actually on time

rf_auc = evaluator.evaluate(rf_pred)

rf_f1 = MulticlassClassificationEvaluator(
    labelCol="IsDelayed", predictionCol="prediction", metricName="f1"
).evaluate(rf_pred)

rf_acc = MulticlassClassificationEvaluator(
    labelCol="IsDelayed", predictionCol="prediction", metricName="accuracy"
).evaluate(rf_pred)

print(f"AUC: {rf_auc}")
print(f"Accuracy: {rf_acc}")
print(f"F1 Score: {rf_f1}")

AUC: 0.6216569285455
Accuracy: 0.8160888573005929
F1 Score: 0.7334454152206201


From the best random forest model we get an AUC of 0.622, which is higher than the linear regression scored. Whilst this may look low, we have to consider that this is only based on pre-departure information. The majority of variance in real-world delay outcomes results from post-departure operational events. If we were to include those post-departure events, it would cause data leakage and ruin our results. The only factor we could have included was weather conditions, but our dataset did not have these and therefore we could not use those either.

In [18]:
# Saving the best model

best_model = cv_model.bestModel
best_model.write().overwrite().save("flights_delay_best_pipeline")

In [19]:
# Inference example on holdout sample
sample = test.limit(1)
display(sample)

prediction = cv_model.transform(sample)
prediction.select("OP_CARRIER", "ORIGIN", "DEST", "CRS_DEP_TIME", "prediction", "probability").show(truncate=False)

DataFrame[ORIGIN: string, DEST: string, FL_DATE: date, OP_CARRIER: string, OP_CARRIER_FL_NUM: int, CRS_DEP_TIME: int, DEP_TIME: float, DEP_DELAY: float, TAXI_OUT: float, WHEELS_OFF: float, WHEELS_ON: float, TAXI_IN: float, CRS_ARR_TIME: int, ARR_TIME: float, ARR_DELAY: float, CANCELLED: float, DIVERTED: float, CRS_ELAPSED_TIME: float, ACTUAL_ELAPSED_TIME: float, AIR_TIME: float, DISTANCE: float, CARRIER_DELAY: float, WEATHER_DELAY: float, NAS_DELAY: float, SECURITY_DELAY: float, LATE_AIRCRAFT_DELAY: float, IsDelayed: int, Year: int, Month: int, ROUTE_FREQ: bigint]

+----------+------+----+------------+----------+----------------------------------------+
|OP_CARRIER|ORIGIN|DEST|CRS_DEP_TIME|prediction|probability                             |
+----------+------+----+------------+----------+----------------------------------------+
|YX        |EWR   |DTW |1715        |0.0       |[0.8065513056309167,0.19344869436908332]|
+----------+------+----+------------+----------+----------------------------------------+



From the inference example we see that the model predicted a 80.7% chance of on-time arrival and a 19.3% chance of delay, classifying the flight as Not Delayed. This confirms that the full preprocessing + modeling pipeline works correctly end-to-end, producing both class probabilities and a final decision output.